# 🎥 Video Analyst — Student 4
**Incident Report Analyzer | CAVIAR CCTV Dataset**

### How to use this notebook
1. Run **Cell 1** — installs libraries (only needed once)
2. Run **Cell 2** — sets up your folder path
3. Run **Cell 3** — generates sample videos (if you have no real videos)
4. Run **Cell 4** — runs the full analyzer → saves CSV + frame images
5. Run **Cell 5** — shows results as a table

> **No video files?** That's fine — Cell 3 creates synthetic CCTV clips for testing.

---
## Cell 1 — Install Libraries

In [6]:
# Run this cell ONCE. Restart kernel if asked.
import sys
!{sys.executable} -m pip install opencv-python numpy pandas --quiet
print('✅ Libraries installed!')

✅ Libraries installed!


---
## Cell 2 — Set Your Project Path

**You must set `PROJECT_ROOT` to wherever you extracted the zip file.**

Examples:
- `C:/Users/Greeshma/Desktop/video_analyst_package`
- `C:/Users/Greeshma/Downloads/video_analyst_package`

In [8]:
import os
from pathlib import Path

# ✏️  CHANGE THIS to where you extracted the zip
PROJECT_ROOT = Path(r"C:/Users/Greeshma/Desktop/video_analyst_package")

# --- Do not change anything below this line ---
VIDEO_DIR  = PROJECT_ROOT / "video"
DATA_DIR   = VIDEO_DIR / "data"
OUTPUT_DIR = VIDEO_DIR / "output"
FRAMES_DIR = OUTPUT_DIR / "frames"
CSV_PATH   = OUTPUT_DIR / "video_analysis.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

print(f'Project root : {PROJECT_ROOT}')
print(f'Video data   : {DATA_DIR}')
print(f'CSV output   : {CSV_PATH}')
print()
if PROJECT_ROOT.exists():
    print('✅ Path found!')
else:
    print('❌ Path NOT found — please update PROJECT_ROOT above')

Project root : C:\Users\Greeshma\Desktop\video_analyst_package
Video data   : C:\Users\Greeshma\Desktop\video_analyst_package\video\data
CSV output   : C:\Users\Greeshma\Desktop\video_analyst_package\video\output\video_analysis.csv

✅ Path found!


---
## Cell 3 — Generate Sample Videos (skip if you have real CAVIAR videos)

In [10]:
import cv2
import numpy as np
import random

CLIPS = [
    "caviar_walk1_cam1.avi",
    "caviar_fight_seq.avi",
    "caviar_left_bag.avi",
    "caviar_fall_corridor.avi",
]

def make_synthetic_clip(filename, seed, fps=25, duration=5, width=320, height=240):
    path   = DATA_DIR / filename
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    writer = cv2.VideoWriter(str(path), fourcc, fps, (width, height), False)
    rng    = random.Random(seed)

    for i in range(fps * duration):
        t     = i / fps
        frame = np.zeros((height, width), dtype=np.uint8)
        noise = np.random.randint(0, 20, (height, width), dtype=np.uint8)
        frame = cv2.add(frame, noise)

        for _ in range(rng.randint(1, 3)):
            x = int(50 + 200 * abs(np.sin(t * 0.8 + rng.random())))
            y = int(40 + 150 * abs(np.cos(t * 0.6 + rng.random())))
            w = rng.randint(20, 60)
            h = rng.randint(30, 80)
            frame[y:y+h, x:x+w] = rng.randint(140, 240)

        cv2.line(frame, (0, height//2), (width, height//2), 60, 1)
        writer.write(frame)

    writer.release()
    print(f'  ✅ Created: {filename}')

print('Generating synthetic CAVIAR-style clips...')
for idx, name in enumerate(CLIPS):
    make_synthetic_clip(name, seed=idx*17+3)

print(f'\nDone! Files saved to: {DATA_DIR}')

Generating synthetic CAVIAR-style clips...
  ✅ Created: caviar_walk1_cam1.avi
  ✅ Created: caviar_fight_seq.avi
  ✅ Created: caviar_left_bag.avi
  ✅ Created: caviar_fall_corridor.avi

Done! Files saved to: C:\Users\Greeshma\Desktop\video_analyst_package\video\data


---
## Cell 4 — Run the Video Analyzer

In [12]:
import cv2
import csv
import uuid
import random
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import Counter

# ── Settings ──────────────────────────────────────────────
SAMPLE_EVERY_N_FRAMES  = 10
MOTION_THRESHOLD       = 500.0
CONFIDENCE_THRESHOLD   = 0.45
SAVE_FRAME_ON_INCIDENT = True

INCIDENT_RULES = [
    ({"fire", "smoke"},      "Fire",            "Critical", 0.70),
    ({"person", "fire"},     "Person in Fire",  "Critical", 0.65),
    ({"person", "knife"},    "Armed Threat",    "Critical", 0.70),
    ({"person", "gun"},      "Armed Threat",    "Critical", 0.70),
    ({"car", "truck"},       "Vehicle Crash",   "High",     0.60),
    ({"motorcycle", "car"},  "Vehicle Crash",   "High",     0.60),
    ({"person", "backpack"}, "Suspicious",      "Medium",   0.55),
    ({"person"},             "Person Detected", "Low",      0.50),
    ({"car"},                "Vehicle",         "Low",      0.50),
]
SEVERITY_RANK = {"Low":1, "Medium":2, "High":3, "Critical":4}
CSV_HEADERS = [
    "Incident_ID","Source_File","Frame_Number","Timestamp_Sec",
    "Detected_Objects","Object_Count","Motion_Score",
    "Video_Event","Severity","Confidence","Frame_Saved","Analyzed_At",
]

# ── Object Detector ────────────────────────────────────────
class ObjectDetector:
    MOCK_POOL = [
        ("person",0.92),("car",0.88),("truck",0.75),("motorcycle",0.80),
        ("backpack",0.65),("fire",0.91),("smoke",0.85),("knife",0.70),
    ]
    def __init__(self):
        self.model = None
        try:
            from ultralytics import YOLO
            self.model = YOLO("yolov8n.pt")
            print("[Detector] YOLOv8 loaded ✅")
        except:
            print("[Detector] Mock mode (ultralytics not installed — results are simulated)")

    def detect(self, frame, frame_number):
        if self.model:
            results = self.model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
            return [{"label":r.names[int(b.cls)],"confidence":round(float(b.conf),3),
                     "bbox":[int(v) for v in b.xyxy[0]]} for r in results for b in r.boxes]
        rng = random.Random(frame_number * 7 + 42)
        picks = rng.sample(self.MOCK_POOL, min(rng.randint(0,3), len(self.MOCK_POOL)))
        return [{"label":l,"confidence":c,"bbox":[50,50,200,300]} for l,c in picks]

# ── Motion Analyzer ────────────────────────────────────────
class MotionAnalyzer:
    def __init__(self):
        self.sub = cv2.createBackgroundSubtractorMOG2(history=200,varThreshold=40,detectShadows=False)
    def score(self, frame):
        return float(np.count_nonzero(self.sub.apply(frame)))

# ── Classifier ────────────────────────────────────────────
def classify(detections, motion):
    labels   = {d["label"].lower() for d in detections}
    avg_conf = sum(d["confidence"] for d in detections)/len(detections) if detections else 0.0
    btype, bsev, bconf = "No Incident", "Low", 0.0
    for req, itype, sev, mconf in INCIDENT_RULES:
        if req.issubset(labels) and avg_conf >= mconf:
            if SEVERITY_RANK.get(sev,0) > SEVERITY_RANK.get(bsev,0):
                btype, bsev, bconf = itype, sev, avg_conf
    if btype == "No Incident" and motion > MOTION_THRESHOLD*3:
        btype, bsev, bconf = "Suspicious Motion", "Medium", 0.55
    return btype, bsev, round(bconf, 3)

# ── Annotate frame ────────────────────────────────────────
def annotate(frame, detections, event, sev):
    out = frame.copy()
    col = {"Critical":(0,0,255),"High":(0,100,255),"Medium":(0,165,255),"Low":(0,200,0)}.get(sev,(180,180,180))
    for d in detections:
        x1,y1,x2,y2 = d["bbox"]
        cv2.rectangle(out,(x1,y1),(x2,y2),col,2)
        cv2.putText(out,f"{d['label']} {d['confidence']:.2f}",(x1,max(y1-6,0)),cv2.FONT_HERSHEY_SIMPLEX,0.5,col,1)
    cv2.rectangle(out,(0,0),(frame.shape[1],28),(30,30,30),-1)
    cv2.putText(out,f"INCIDENT: {event}  [{sev}]",(8,19),cv2.FONT_HERSHEY_SIMPLEX,0.6,col,2)
    return out

# ── Process one video ─────────────────────────────────────
def process_video(video_path, detector, motion_analyzer):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  ❌ Cannot open: {video_path.name}"); return []
    fps   = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w,h   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"\n  📹 {video_path.name}  {w}x{h} {fps:.0f}fps {total} frames")

    rows, fidx, now = [], 0, datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    while True:
        ret, frame = cap.read()
        if not ret: break
        if fidx % SAMPLE_EVERY_N_FRAMES == 0:
            ts       = round(fidx/fps, 3)
            dets     = detector.detect(frame, fidx)
            motion   = motion_analyzer.score(frame)
            evt,sev,conf = classify(dets, motion)
            lstr     = "; ".join(d["label"] for d in dets) or "none"
            fsaved   = ""
            if SAVE_FRAME_ON_INCIDENT and evt not in ("No Incident","Person Detected"):
                ann  = annotate(frame, dets, evt, sev)
                fn   = f"{video_path.stem}_f{fidx:06d}_{evt.replace(' ','_')}.jpg"
                cv2.imwrite(str(FRAMES_DIR/fn), ann)
                fsaved = fn
            rows.append({"Incident_ID":str(uuid.uuid4())[:8].upper(),
                         "Source_File":video_path.name,"Frame_Number":fidx,
                         "Timestamp_Sec":ts,"Detected_Objects":lstr,
                         "Object_Count":len(dets),"Motion_Score":round(motion,1),
                         "Video_Event":evt,"Severity":sev,"Confidence":conf,
                         "Frame_Saved":fsaved,"Analyzed_At":now})
            if evt != "No Incident":
                print(f"    [{fidx:5d}] {ts:6.2f}s  {evt:<20} {sev:<8}  {lstr}")
        fidx += 1
    cap.release()
    print(f"  ✅ {fidx} frames → {len(rows)} samples")
    return rows

# ── MAIN ──────────────────────────────────────────────────
print("="*55)
print("  Video Analyst — Student 4")
print(f"  Data  : {DATA_DIR}")
print(f"  Output: {CSV_PATH}")
print("="*55)

exts = {".mp4",".avi",".mov",".mkv",".wmv",".flv"}
video_files = [p for p in DATA_DIR.iterdir() if p.suffix.lower() in exts]

detector = ObjectDetector()
motion   = MotionAnalyzer()
all_rows = []

if not video_files:
    print("  No video files in data/ — run Cell 3 first to generate sample videos!")
else:
    for vf in sorted(video_files):
        all_rows.extend(process_video(vf, detector, motion))

    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_HEADERS)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n  💾 CSV saved: {CSV_PATH}  ({len(all_rows)} rows)")

    events = Counter(r["Video_Event"] for r in all_rows)
    sev    = Counter(r["Severity"]    for r in all_rows)
    print("\n  Events:", dict(events.most_common()))
    print("  Severity:", dict(sev))

  Video Analyst — Student 4
  Data  : C:\Users\Greeshma\Desktop\video_analyst_package\video\data
  Output: C:\Users\Greeshma\Desktop\video_analyst_package\video\output\video_analysis.csv
[Detector] Mock mode (ultralytics not installed — results are simulated)

  📹 caviar_fall_corridor.avi  320x240 25fps 125 frames
    [    0]   0.00s  Suspicious Motion    Medium    none
    [   10]   0.40s  Suspicious Motion    Medium    backpack; fire; motorcycle
    [   20]   0.80s  Fire                 Critical  fire; car; smoke
    [   30]   1.20s  Suspicious Motion    Medium    none
    [   40]   1.60s  Vehicle Crash        High      truck; person; car
    [   50]   2.00s  Suspicious Motion    Medium    knife; backpack; fire
    [   60]   2.40s  Suspicious Motion    Medium    none
    [   70]   2.80s  Suspicious Motion    Medium    truck
    [   80]   3.20s  Suspicious Motion    Medium    truck; knife; backpack
    [   90]   3.60s  Suspicious Motion    Medium    none
    [  100]   4.00s  Suspiciou

---
## Cell 5 — View Results

In [14]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f'Total rows: {len(df)}')
print(f'Columns   : {list(df.columns)}')
print()

# Show incidents only (skip No Incident rows)
incidents = df[df['Video_Event'] != 'No Incident']
print(f'Incidents detected: {len(incidents)}')
display(incidents[['Incident_ID','Source_File','Timestamp_Sec','Video_Event','Severity','Confidence','Detected_Objects']].head(20))

Total rows: 52
Columns   : ['Incident_ID', 'Source_File', 'Frame_Number', 'Timestamp_Sec', 'Detected_Objects', 'Object_Count', 'Motion_Score', 'Video_Event', 'Severity', 'Confidence', 'Frame_Saved', 'Analyzed_At']

Incidents detected: 50


,Incident_ID,Source_File,Timestamp_Sec,Video_Event,Severity,Confidence,Detected_Objects
0,B7F145D5,caviar_fall_corridor.avi,0.0,Suspicious Motion,Medium,0.55,none
1,F16F6C75,caviar_fall_corridor.avi,0.4,Suspicious Motion,Medium,0.55,backpack; fire; motorcycle
2,8B37725B,caviar_fall_corridor.avi,0.8,Fire,Critical,0.88,fire; car; smoke
3,CDD3247D,caviar_fall_corridor.avi,1.2,Suspicious Motion,Medium,0.55,none
4,E4C58FD0,caviar_fall_corridor.avi,1.6,Vehicle Crash,High,0.85,truck; person; car
5,D37666BA,caviar_fall_corridor.avi,2.0,Suspicious Motion,Medium,0.55,knife; backpack; fire
6,2B3C55BC,caviar_fall_corridor.avi,2.4,Suspicious Motion,Medium,0.55,none
7,AD5336B4,caviar_fall_corridor.avi,2.8,Suspicious Motion,Medium,0.55,truck
8,6512F3F5,caviar_fall_corridor.avi,3.2,Suspicious Motion,Medium,0.55,truck; knife; backpack
9,A03DD011,caviar_fall_corridor.avi,3.6,Suspicious Motion,Medium,0.55,none


---
## Cell 6 — Copy output to project integration folder (optional)

In [16]:
import shutil

# If your full project repo is cloned separately, set this path:
REPO_ROOT = Path(r"C:/Users/Greeshma/incident-report-analyzer")
dest = REPO_ROOT / "video" / "output" / "video_analysis.csv"

if REPO_ROOT.exists():
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(CSV_PATH, dest)
    print(f'✅ Copied to repo: {dest}')
else:
    print('ℹ️  Repo path not found — skipping copy. Your CSV is at:')
    print(f'   {CSV_PATH}')

ℹ️  Repo path not found — skipping copy. Your CSV is at:
   C:\Users\Greeshma\Desktop\video_analyst_package\video\output\video_analysis.csv
